# 01 — Cleaning and validation (v2)

**Input:** `Basal v2.xlsx`  
**Outputs:** `dataset_clean.csv`, `data_errors.csv`, `error_examples.csv`

Clean numeric formats, validate and correct deterministic derived variables, identify inconsistencies, and record exact matches across the seven original HRV measures. Preserve `redcap_repeat_instrument` even when empty. **All participants remain in the dataset**; no age filtering or categorization. Ambiguous HR/RR differences and identical HRV profiles are reported without changing measurements.

Uses `pandas` and `numpy` only, plus Python's standard `pathlib`. Place `Basal v2.xlsx` beside the notebook or edit `INPUT_FILE`.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

INPUT_FILE = Path("../../raw_data/Basal v2.xlsx")
OUTPUT_DIR = Path("../../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

raw = pd.read_excel(INPUT_FILE)
df = raw.copy()
df.insert(0, "source_excel_row", np.arange(2, len(df) + 2))

print(f"Input: {INPUT_FILE} | {len(raw)} rows | {raw.shape[1]} columns")
display(pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique": raw.nunique(dropna=True),
}))

Input: ../../raw_data/Basal v2.xlsx | 530 rows | 20 columns


,dtype,missing,unique
Unnamed: 0,float64,530,0
Unnamed: 1,float64,530,0
redcap_repeat_instrument,float64,530,0
sex,int64,0,2
age,int64,0,35
t2m_pre_mean_rr,float64,0,303
t2m_pre_mean_hr,float64,1,55
t2m_pre_sdnn,float64,11,268
t2m_pre_rmssd,object,8,278
t2m_pre_hf,int64,0,290


In [2]:
# One entry per anomaly or correction. Excel row numbers start at 2 (header = row 1).
issues = []

def flag(row, column, issue, original, processed, action, detail=""):
    issues.append({
        "source_excel_row": row,
        "column": column,
        "issue": issue,
        "original_value": original,
        "processed_value": processed,
        "action": action,
        "detail": detail,
    })

# Drop empty unnamed export columns, but preserve the original REDCap metadata.
metadata_columns = ["redcap_repeat_instrument"]
empty_columns = [col for col in raw.columns[raw.isna().all()] if col not in metadata_columns]
for col in empty_columns:
    flag(pd.NA, col, "empty_column", pd.NA, pd.NA,
         "removed_column", "All values missing")
df = df.drop(columns=empty_columns)

for col in metadata_columns:
    if col in raw.columns and raw[col].isna().all():
        flag(pd.NA, col, "empty_metadata_column", pd.NA, pd.NA,
             "retained_for_traceability", "Column is empty in the supplied Excel file; values were not recovered or inferred")

# Follow the names used in the previous preprocessing notebook.
df = df.rename(columns={
    "weight kg": "weight_kg",
    "height cm": "height_cm",
    "height mt": "height_m",
})

print("Removed fully empty export columns:", empty_columns)
print("Metadata columns retained:", [col for col in metadata_columns if col in df])
print("Remaining columns:", df.columns.tolist())


Removed fully empty export columns: ['Unnamed: 0', 'Unnamed: 1']
Metadata columns retained: ['redcap_repeat_instrument']
Remaining columns: ['source_excel_row', 'redcap_repeat_instrument', 'sex', 'age', 't2m_pre_mean_rr', 't2m_pre_mean_hr', 't2m_pre_sdnn', 't2m_pre_rmssd', 't2m_pre_hf', 't2m_pre_lf', 't2m_pre_vlf', 'bp_systolic', 'bp_diastolic', 'bp_pam', 'bp_pp', 'imc', 'weight_kg', 'height_cm', 'height_m']


In [3]:
# Convert the known quantitative columns; record original text values and failures.
numeric_columns = [
    "sex", "age", "t2m_pre_mean_rr", "t2m_pre_mean_hr",
    "t2m_pre_sdnn", "t2m_pre_rmssd", "t2m_pre_hf", "t2m_pre_lf",
    "t2m_pre_vlf", "bp_systolic", "bp_diastolic", "bp_pam",
    "bp_pp", "imc", "weight_kg", "height_cm", "height_m",
]

for col in numeric_columns:
    if col not in df:
        flag(pd.NA, col, "missing_expected_column", pd.NA, pd.NA,
             "review", "Expected column not found")
        continue

    old = df[col].copy()
    if old.dtype == "object":
        normalized = old.astype("string").str.replace(r"\s+", "", regex=True).str.replace(",", ".", regex=False)
        new = pd.to_numeric(normalized, errors="coerce")
    else:
        new = pd.to_numeric(old, errors="coerce")

    for idx in df.index:
        original = old.loc[idx]
        converted = new.loc[idx]
        row = int(df.loc[idx, "source_excel_row"])
        if isinstance(original, str) and original.strip():
            if pd.isna(converted):
                flag(row, col, "unparseable_number", original, pd.NA,
                     "set_missing_review", "Cannot interpret original text as a number")
            else:
                flag(row, col, "text_number", original, converted,
                     "converted_to_numeric", "Original Excel cell was text")
        elif pd.notna(original) and pd.isna(converted):
            flag(row, col, "unparseable_number", original, pd.NA,
                 "set_missing_review", "Numeric conversion failed")

    df[col] = new

print("Numeric columns converted:", [c for c in numeric_columns if c in df])

Numeric columns converted: ['sex', 'age', 't2m_pre_mean_rr', 't2m_pre_mean_hr', 't2m_pre_sdnn', 't2m_pre_rmssd', 't2m_pre_hf', 't2m_pre_lf', 't2m_pre_vlf', 'bp_systolic', 'bp_diastolic', 'bp_pam', 'bp_pp', 'imc', 'weight_kg', 'height_cm', 'height_m']


In [4]:
# Check height (cm and m) and BMI. As in the original notebook,
# height_cm is the reference when both height columns are available.
if {"height_cm", "height_m"}.issubset(df.columns):
    height_from_cm = df["height_cm"] / 100
    comparable = (df["height_cm"] > 0) & (df["height_m"] > 0)
    inconsistent = comparable & ((df["height_m"] - height_from_cm).abs() > 0.01)
    recover_m = (df["height_cm"] > 0) & df["height_m"].isna()
    recover_cm = df["height_cm"].isna() & (df["height_m"] > 0)

    for idx in df.index[inconsistent | recover_m]:
        row = int(df.at[idx, "source_excel_row"])
        old = df.at[idx, "height_m"]
        new = height_from_cm.loc[idx]
        flag(row, "height_m", "height_disagreement" if pd.notna(old) else "missing_derived_height",
             old, new, "corrected_from_height_cm",
             f"height_cm={df.at[idx, 'height_cm']}; height_cm used as reference")
        df.at[idx, "height_m"] = new

    for idx in df.index[recover_cm]:
        row = int(df.at[idx, "source_excel_row"])
        new = df.at[idx, "height_m"] * 100
        flag(row, "height_cm", "missing_derived_height", pd.NA, new,
             "filled_from_height_m", "height_cm missing; height_m available")
        df.at[idx, "height_cm"] = new

if {"weight_kg", "height_m", "imc"}.issubset(df.columns):
    eligible = (df["weight_kg"] > 0) & (df["height_m"] > 0)
    bmi_calculated = df["weight_kg"] / df["height_m"].pow(2)
    bmi_difference = eligible & df["imc"].notna() & ((df["imc"] - bmi_calculated).abs() > 0.10)
    bmi_missing = eligible & df["imc"].isna()

    for idx in df.index[bmi_difference | bmi_missing]:
        row = int(df.at[idx, "source_excel_row"])
        old = df.at[idx, "imc"]
        new = bmi_calculated.loc[idx]
        flag(row, "imc", "bmi_disagreement" if pd.notna(old) else "missing_bmi",
             old, new, "recomputed_from_weight_and_height",
             "BMI = weight_kg / height_m^2; tolerance 0.10 kg/m^2")
        df.at[idx, "imc"] = new

print("Height and BMI checks completed")

Height and BMI checks completed


In [5]:
# Recompute deterministic blood pressure derivatives from SBP and DBP.
if {"bp_systolic", "bp_diastolic"}.issubset(df.columns):
    valid_bp = (df["bp_systolic"] > 0) & (df["bp_diastolic"] > 0) & (df["bp_systolic"] > df["bp_diastolic"])
    derived_bp = {
        "bp_pp": (df["bp_systolic"] - df["bp_diastolic"], 0.01),
        "bp_pam": (df["bp_diastolic"] + (df["bp_systolic"] - df["bp_diastolic"]) / 3, 0.10),
    }
    for col, (calculated, tolerance) in derived_bp.items():
        if col not in df:
            continue
        mismatch = valid_bp & df[col].notna() & ((df[col] - calculated).abs() > tolerance)
        missing = valid_bp & df[col].isna()
        for idx in df.index[mismatch | missing]:
            row = int(df.at[idx, "source_excel_row"])
            old = df.at[idx, col]
            new = calculated.loc[idx]
            flag(row, col, "derived_bp_disagreement" if pd.notna(old) else "missing_derived_bp",
                 old, new, "recomputed_from_sbp_dbp", f"Tolerance: {tolerance} mmHg")
            df.at[idx, col] = new

print("Derived blood pressure checks completed")

Derived blood pressure checks completed


In [6]:
# Missing data, review-only ranges, HR/RR mismatches, and identical HRV profiles.
# These findings do not trigger participant exclusion or automatic changes to HRV.
for col in numeric_columns:
    if col not in df:
        continue
    missing = df[col].isna()
    for idx in df.index[missing]:
        original_name = {"weight_kg": "weight kg", "height_cm": "height cm", "height_m": "height mt"}.get(col, col)
        flag(int(df.at[idx, "source_excel_row"]), col, "missing_value",
             raw.at[idx, original_name] if original_name in raw else pd.NA,
             pd.NA, "kept_missing", "Not imputed")

# Basic screening limits from the earlier notebook. Flags are not exclusions.
review_ranges = {
    "age": (18, 100), "weight_kg": (30, 250), "height_cm": (120, 230),
    "height_m": (1.2, 2.3), "imc": (10, 80),
    "bp_systolic": (70, 250), "bp_diastolic": (40, 150),
    "t2m_pre_mean_hr": (30, 220), "t2m_pre_mean_rr": (250, 2000),
    "t2m_pre_sdnn": (0, 500), "t2m_pre_rmssd": (0, 500),
    "t2m_pre_hf": (0, 100000), "t2m_pre_lf": (0, 100000),
    "t2m_pre_vlf": (0, 100000),
}
for col, (low, high) in review_ranges.items():
    if col not in df:
        continue
    mask = df[col].notna() & ((df[col] < low) | (df[col] > high))
    for idx in df.index[mask]:
        value = df.at[idx, col]
        flag(int(df.at[idx, "source_excel_row"]), col, "outside_review_range",
             value, value, "review_no_change",
             f"Screening interval [{low}, {high}]; not an exclusion criterion")

if {"bp_systolic", "bp_diastolic"}.issubset(df.columns):
    mask = (df["bp_systolic"].notna() & df["bp_diastolic"].notna()
            & (df["bp_systolic"] <= df["bp_diastolic"]))
    for idx in df.index[mask]:
        flag(int(df.at[idx, "source_excel_row"]), "bp_systolic/bp_diastolic",
             "invalid_bp_order", df.at[idx, "bp_systolic"], df.at[idx, "bp_diastolic"],
             "review_no_change", "Systolic BP must be greater than diastolic BP")

# A near-10-fold difference is reported separately; neither HR nor RR is auto-corrected.
if {"t2m_pre_mean_hr", "t2m_pre_mean_rr"}.issubset(df.columns):
    hr = df["t2m_pre_mean_hr"]
    rr = df["t2m_pre_mean_rr"]
    valid = (hr > 0) & (rr > 0)
    estimated_hr = 60000 / rr
    different = valid & ((hr - estimated_hr).abs() > 2)
    ratio = (hr * rr) / 60000
    factor_ten = pd.Series(
        np.isclose(ratio, 0.1, rtol=0.03) | np.isclose(ratio, 10, rtol=0.03),
        index=df.index,
    )
    for idx in df.index[different]:
        factor = bool(factor_ten.loc[idx])
        issue = "hr_rr_factor_10_pattern" if factor else "hr_rr_disagreement"
        detail = (f"Observed HR={hr.loc[idx]:g} bpm and RR={rr.loc[idx]:g} ms; "
                  f"60,000/RR={estimated_hr.loc[idx]:.3f} bpm; "
                  f"HR*RR/60,000={ratio.loc[idx]:.4f}; "
                  + ("near 10-fold inconsistency; verify source before changing either value"
                     if factor else "difference above 2 bpm; verify source"))
        flag(int(df.at[idx, "source_excel_row"]), "t2m_pre_mean_hr/t2m_pre_mean_rr",
             issue, hr.loc[idx], hr.loc[idx], "review_no_change", detail)

# Same seven HRV measurements across different records: flag each row in the group.
hrv_columns = [
    "t2m_pre_mean_rr", "t2m_pre_mean_hr", "t2m_pre_sdnn", "t2m_pre_rmssd",
    "t2m_pre_hf", "t2m_pre_lf", "t2m_pre_vlf",
]
if set(hrv_columns).issubset(df.columns):
    complete = df[hrv_columns].notna().all(axis=1)
    hrvs = df.loc[complete]
    repeated = hrvs.duplicated(subset=hrv_columns, keep=False)
    for group_num, (_, group) in enumerate(
        hrvs.loc[repeated].groupby(hrv_columns, sort=False), start=1
    ):
        matching_rows = sorted(group["source_excel_row"].astype(int).tolist())
        for row in matching_rows:
            flag(row, "hrv_profile", "identical_hrv_profile",
                 "Seven HRV variables match another row", "Unchanged",
                 "review_no_change",
                 f"HRV match group {group_num}; Excel rows: {', '.join(map(str, matching_rows))}")

# Exact duplicate full records are a separate issue from identical HRV profiles.
original_nonempty = raw.drop(columns=empty_columns)
for idx in raw.index[original_nonempty.duplicated(keep=False)]:
    flag(int(df.at[idx, "source_excel_row"]), "all_columns", "possible_duplicate_record",
         pd.NA, pd.NA, "review_no_change", "Exact duplicate of another raw row")

print("Checks completed; no rows removed")


Checks completed; no rows removed


In [7]:
# Exactly three CSV outputs. Error examples contain original rows, not reconstructed data.
errors = pd.DataFrame(issues, columns=[
    "source_excel_row", "column", "issue", "original_value",
    "processed_value", "action", "detail",
])

rows_with_issues = errors["source_excel_row"].dropna().astype(int).unique()
examples = raw.iloc[rows_with_issues - 2].copy()
examples.insert(0, "source_excel_row", rows_with_issues)
issue_list = errors.dropna(subset=["source_excel_row"]).groupby("source_excel_row")["issue"].apply(
    lambda items: "; ".join(sorted(set(items)))
)
examples["detected_issues"] = examples["source_excel_row"].map(issue_list)

# Show the corrected counterparts for the variables most likely to change.
for col in ["t2m_pre_rmssd", "height_m", "imc"]:
    if col in df:
        examples[f"processed_{col}"] = examples["source_excel_row"].map(df.set_index("source_excel_row")[col])

errors.to_csv(OUTPUT_DIR / "data_errors.csv", index=False)
df.to_csv(OUTPUT_DIR / "dataset_clean.csv", index=False)
examples.to_csv(OUTPUT_DIR / "error_examples.csv", index=False)

print(f"Raw rows: {len(raw)} | Processed rows: {len(df)} | Columns: {df.shape[1]}")
print(f"Issues recorded: {len(errors)} | Rows with issues: {len(examples)}")
print("\nIssues by type:")
display(errors.groupby(["issue", "action"]).size().rename("count").to_frame())
print("\nOutput files:")
for name in ["dataset_clean.csv", "data_errors.csv", "error_examples.csv"]:
    print(OUTPUT_DIR / name)

display(df.head(5))

Raw rows: 530 | Processed rows: 530 | Columns: 19
Issues recorded: 183 | Rows with issues: 155

Issues by type:


,,count
issue,action,
bmi_disagreement,recomputed_from_weight_and_height,9
empty_column,removed_column,2
empty_metadata_column,retained_for_traceability,1
height_disagreement,corrected_from_height_cm,9
hr_rr_disagreement,review_no_change,4
hr_rr_factor_10_pattern,review_no_change,11
identical_hrv_profile,review_no_change,16
missing_value,kept_missing,20
outside_review_range,review_no_change,14



Output files:
../../outputs/dataset_clean.csv
../../outputs/data_errors.csv
../../outputs/error_examples.csv


,source_excel_row,redcap_repeat_instrument,sex,age,t2m_pre_mean_rr,t2m_pre_mean_hr,t2m_pre_sdnn,t2m_pre_rmssd,t2m_pre_hf,t2m_pre_lf,t2m_pre_vlf,bp_systolic,bp_diastolic,bp_pam,bp_pp,imc,weight_kg,height_cm,height_m
0,2,NaN,1,75,748.0,80.0,308.0,21.7,222,713,172,108,70,82.666667,38,27.343750,70.0,160,1.60
1,3,NaN,1,66,953.0,63.0,255.0,18.8,87,226,29,122,72,88.666667,50,23.422368,52.0,149,1.49
2,4,NaN,2,77,877.0,68.0,147.0,11.7,26,137,9,125,85,98.333333,40,28.027682,81.0,170,1.70
3,5,NaN,1,77,819.0,73.0,170.0,17.5,118,130,14,126,72,90.000000,54,33.203125,85.0,160,1.60
4,6,NaN,1,73,846.0,71.0,176.0,13.6,66,66,18,130,80,96.666667,50,30.261831,69.0,151,1.51
